# Lab 04: Fine-Tune an Instruction Model with LoRA

This practical fine-tunes a small instruction-following model to answer computer-shop questions. Run the notebook in Google Colab with a GPU runtime. Before the first run, choose Runtime > Disconnect and delete runtime, reconnect with a T4 GPU, and run all cells from the top. This clears stale CPU-only bitsandbytes libraries. The notebook verifies the NF4 CUDA kernel before downloading the model, then uses Qwen2.5-0.5B-Instruct when at least 8 GiB VRAM is available, otherwise SmolLM2-360M-Instruct.


## 1. Install libraries and inspect hardware
The GPU accelerates the large matrix multiplications used in transformer training and provides fast memory for model weights, activations, and optimizer state. The first code cell records the GPU model and memory; save its output in the Lab README after running.

In [ ]:
# Use a clean Colab T4 runtime. Remove any stale CPU-only bitsandbytes install,
# then install the official PyPI wheel without replacing Colab's CUDA-enabled torch.
%pip -q install --upgrade --no-cache-dir "transformers==5.23.0" "datasets==5.2.0" "peft==0.22.0" "trl==1.18.2" "accelerate==1.16.0" "pyarrow==23.0.1" "huggingface_hub>=1.32.0,<3.0" "gradio>=6.31.0,<7.0"
%pip -q uninstall -y bitsandbytes
%pip -q install --no-cache-dir --no-deps --index-url https://pypi.org/simple "bitsandbytes==0.50.2"

import torch
from IPython.display import display

if not torch.cuda.is_available():
    raise RuntimeError("No CUDA GPU detected. Select Runtime > Change runtime type > T4 GPU, reconnect, then run all cells from the top.")
gpu_name = torch.cuda.get_device_name(0)
gpu_memory_gib = torch.cuda.get_device_properties(0).total_memory / 1024**3
gpu_capability = torch.cuda.get_device_capability(0)
supports_bf16 = gpu_capability >= (8, 0) and torch.cuda.is_bf16_supported()
if gpu_capability < (6, 0):
    raise RuntimeError(f"This notebook uses NF4 QLoRA, which needs CUDA compute capability 6.0 or newer; detected {gpu_capability}. Select a T4 GPU.")
use_qlora = True
print(f"GPU: {gpu_name}; memory: {gpu_memory_gib:.2f} GiB; compute capability: {gpu_capability}")
MODEL_ID = "Qwen/Qwen2.5-0.5B-Instruct" if gpu_memory_gib >= 8 else "HuggingFaceTB/SmolLM2-360M-Instruct"
print("Model:", MODEL_ID)
print("Method:", "4-bit NF4 QLoRA" if use_qlora else "standard LoRA")
print("GPU speeds up the parallel matrix operations and high-bandwidth memory work used during LLM fine-tuning.")

# Confirm a CUDA-native bitsandbytes backend and run an NF4 round-trip before model download.
import bitsandbytes as bnb
try:
    cuda_versions = bnb.cextension.get_available_cuda_binary_versions()
    if not bnb.cextension.lib.compiled_with_cuda:
        raise RuntimeError(f"CPU-only native library loaded; packaged CUDA binaries: {cuda_versions}")
    probe = torch.randn((128, 128), device="cuda", dtype=torch.float16)
    packed, quant_state = bnb.functional.quantize_4bit(probe, quant_type="nf4")
    restored = bnb.functional.dequantize_4bit(packed, quant_state)
    assert restored.shape == probe.shape and torch.isfinite(restored).all()
    del probe, packed, quant_state, restored
    torch.cuda.empty_cache()
    print(f"bitsandbytes {bnb.__version__} NF4 CUDA preflight: passed; CUDA binaries: {cuda_versions}")
except Exception as exc:
    raise RuntimeError("bitsandbytes did not load a working CUDA NF4 kernel. Stop here before model download. Use Runtime > Disconnect and delete runtime, reconnect with T4 GPU, and run all cells from the top. If this repeats, capture torch.version.cuda and the bitsandbytes CUDA binary list for diagnosis.") from exc

## 2. Create a small domain dataset
The 36 examples below are original computer-shop support questions and answers. The final eight rows are held out for evaluation and are not used for training. The separate prompts at the end are also absent from the training set.

In [ ]:
from datasets import Dataset
from transformers import AutoTokenizer

qa_pairs = [
('What does a computer shop sell?', 'A computer shop sells computers, components, peripherals, and related services.'),
('Can I get help choosing a laptop?', 'Yes. Tell us your budget and what you plan to use it for, and we can suggest suitable laptops.'),
('What is the difference between RAM and storage?', 'RAM is short-term working memory; storage keeps files and programs when the computer is off.'),
('Do you sell desktop computers?', 'Yes. We sell desktops for everyday use, study, creative work, and gaming.'),
('Can you upgrade my laptop memory?', 'We can check whether your laptop memory is upgradeable and quote the compatible RAM.'),
('How do I choose a graphics card?', 'Choose a graphics card based on your software, monitor resolution, power supply, case space, and budget.'),
('Do you install components?', 'Yes. We can install compatible components and test the computer after installation.'),
('What should I bring for a repair?', 'Bring the device, its charger, and a description of the problem. Back up important files first if possible.'),
('Can you recover deleted files?', 'We can assess the storage device, but recovery is not guaranteed. Stop using the device to avoid overwriting data.'),
('What is an SSD?', 'An SSD is a fast storage device with no moving parts, commonly used for operating systems and files.'),
('Do you sell keyboards?', 'Yes. We offer wired and wireless keyboards in several layouts and styles.'),
('Can I return an item?', 'Returns depend on the store policy and the item condition. Keep the receipt and contact the shop to confirm eligibility.'),
('How can I check whether a charger is compatible?', 'Match the device model, connector, voltage, and required wattage before using a charger.'),
('Do you provide a warranty?', 'Warranty coverage depends on the product and manufacturer. Check the product documents or ask us for its warranty terms.'),
('My computer is overheating. What should I do?', 'Turn it off if it becomes very hot, keep vents clear, and arrange a diagnostic check if overheating continues.'),
('Can you remove a virus?', 'We can inspect the computer and help remove malicious software. Avoid entering passwords on a device you suspect is infected.'),
('What is a motherboard?', 'The motherboard is the main circuit board that connects the processor, memory, storage, and other components.'),
('Do you build custom PCs?', 'Yes. We can help select compatible parts and assemble a custom PC for your workload and budget.'),
('How much storage should I get?', 'It depends on your files and applications. A larger SSD is useful for games, media, and creative projects.'),
('Can I connect two monitors?', 'Many computers support two monitors, but it depends on the graphics hardware and available display ports.'),
('What is a power supply unit?', 'A power supply unit converts wall power into the voltages computer components need.'),
('My laptop battery drains quickly. Can you help?', 'We can check battery health, power settings, and background apps, then advise whether the battery needs replacement.'),
('Do you sell webcams?', 'Yes. We sell USB webcams for video calls, classes, and streaming.'),
('How do I protect my files?', 'Keep a separate backup on an external drive or trusted cloud service and update it regularly.'),
('Can you set up a new computer?', 'We can help with initial setup, updates, account sign-in, and transferring files from an old computer.'),
('What is thermal paste?', 'Thermal paste helps transfer heat from the processor to its cooler.'),
('Can I use a gaming PC for school?', 'Yes. A gaming PC can handle schoolwork, though it may cost more and use more power than needed for basic tasks.'),
('Do you sell USB-C hubs?', 'Yes. Check that a hub supports your device and the ports or display features you need.'),
('Why is my computer slow?', 'Common causes include too many startup apps, low storage, limited memory, malware, or aging hardware. A diagnostic can identify the cause.'),
('Can you clean dust from a desktop?', 'Yes. We can clean internal dust safely and check that the fans are working properly.'),
('What is a CPU?', 'The CPU is the processor that executes instructions and coordinates many computer tasks.'),
('Should I shut down my computer every night?', 'Either shutdown or sleep is fine for most users. Restarting occasionally also helps apply updates.'),
('Can I add Wi-Fi to a desktop?', 'Usually. A compatible Wi-Fi card or USB adapter can add wireless networking.'),
('What information do you need to recommend a PC?', 'Share your budget, main applications, preferred screen size, and any portability or gaming needs.'),
('Do you repair printers?', 'We can advise on common setup issues and let you know whether printer repairs are available at the shop.'),
('How do I safely transport a desktop PC?', 'Shut it down, unplug cables, secure loose accessories, and protect the case from impact and moisture.'),
]
assert len(qa_pairs) >= 30
train_pairs, eval_pairs = qa_pairs[:-8], qa_pairs[-8:]
print(f"Q&A pairs: {len(qa_pairs)}; train: {len(train_pairs)}; held-out: {len(eval_pairs)}")
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
SYSTEM_PROMPT = "You are a helpful computer-shop assistant. Be concise, accurate, and do not invent store policies."
def make_prompt(question):
    messages = [{"role":"system","content":SYSTEM_PROMPT},{"role":"user","content":question}]
    return tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
train_ds = Dataset.from_dict({"prompt":[make_prompt(q) for q,a in train_pairs], "completion":[a + tokenizer.eos_token for q,a in train_pairs]})
eval_ds = Dataset.from_dict({"prompt":[make_prompt(q) for q,a in eval_pairs], "completion":[a + tokenizer.eos_token for q,a in eval_pairs]})
print("Formatted prompt-completion example:\n", train_ds[0]["prompt"], train_ds[0]["completion"])

## 3. Compare the original model and configure 4-bit QLoRA
This run uses a T4-compatible current bitsandbytes wheel and 4-bit NF4 quantization. The dataset uses prompt/completion columns, so TRL computes supervised loss on the answer completion rather than the prompt.

In [ ]:
import torch
from transformers import AutoModelForCausalLM, BitsAndBytesConfig
from peft import LoraConfig, PeftModel, prepare_model_for_kbit_training

compute_dtype = torch.bfloat16 if supports_bf16 else torch.float16
quant_config = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4", bnb_4bit_use_double_quant=True, bnb_4bit_compute_dtype=compute_dtype)
base_model = AutoModelForCausalLM.from_pretrained(MODEL_ID, quantization_config=quant_config, torch_dtype=compute_dtype, device_map={"": 0})
base_model.config.use_cache = True

def ask(model, question, max_new_tokens=100):
    inputs = tokenizer.apply_chat_template([{"role":"system","content":SYSTEM_PROMPT},{"role":"user","content":question}], add_generation_prompt=True, return_tensors="pt", return_dict=True)
    inputs = {k:v.to(model.device) for k,v in inputs.items()}
    prompt_len = inputs["input_ids"].shape[-1]
    was_training = model.training
    model.eval()
    with torch.inference_mode():
        out = model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False, pad_token_id=tokenizer.pad_token_id, eos_token_id=tokenizer.eos_token_id)
    model.train(was_training)
    return tokenizer.decode(out[0][prompt_len:], skip_special_tokens=True).strip()

heldout_questions = [q for q,a in eval_pairs]
new_questions = ["A customer needs a quiet computer for online classes on a tight budget. What should we ask before recommending one?", "A desktop powers on but shows no picture. What safe basic checks can the customer try?", "Can you guarantee recovery of every deleted file?"]
original_answers = [ask(base_model,q) for q in heldout_questions]
original_new_answers = [ask(base_model,q) for q in new_questions]
base_model = prepare_model_for_kbit_training(base_model, use_gradient_checkpointing=True, gradient_checkpointing_kwargs={"use_reentrant":False})
base_model.config.use_cache = False
lora_config = LoraConfig(r=8, lora_alpha=16, lora_dropout=0.05, bias="none", task_type="CAUSAL_LM", target_modules=["q_proj","k_proj","v_proj","o_proj","gate_proj","up_proj","down_proj"])
print(f"Base parameters: {sum(p.numel() for p in base_model.parameters()):,}")

## 4. Supervised fine-tuning with TRL SFTTrainer
Training is intentionally short for this small teaching dataset. Increase epochs only after checking training time and overfitting.

In [ ]:
from trl import SFTConfig, SFTTrainer

training_args = SFTConfig(
    output_dir="computer-shop-lora-checkpoints", num_train_epochs=3,
    per_device_train_batch_size=2, per_device_eval_batch_size=2,
    gradient_accumulation_steps=4, learning_rate=1e-4, logging_steps=1,
    eval_strategy="epoch", save_strategy="no", report_to="none", max_length=512,
    completion_only_loss=True, gradient_checkpointing=True,
    gradient_checkpointing_kwargs={"use_reentrant":False},
    # Keep Trainer AMP disabled for consistent T4 CUDA gradients. NF4 weights still
    # compute in FP16 via BitsAndBytesConfig above, while LoRA gradients stay FP32.
    fp16=False, bf16=False, optim="paged_adamw_8bit",
)
trainer = SFTTrainer(model=base_model, args=training_args, train_dataset=train_ds, eval_dataset=eval_ds, processing_class=tokenizer, peft_config=lora_config)
trainer.model.print_trainable_parameters()
train_result = trainer.train()
print(train_result)
print("Training/evaluation history:", trainer.state.log_history)

## 5. Evaluate, compare held-out prompts, and save the adapter
The held-out questions were not part of training. We report TRL evaluation loss and perplexity with loss restricted to answer completions, and compare original versus fine-tuned answers using the exact same prompts. A small custom dataset demonstrates the workflow but does not guarantee factual or production-quality behavior.


In [ ]:
import math, json, shutil
import pandas as pd
from pathlib import Path
from peft import PeftModel

metrics = trainer.evaluate()
eval_loss = float(metrics["eval_loss"])
perplexity = math.exp(eval_loss) if eval_loss < 20 else float("inf")
print("Evaluation metrics:", metrics)
print(f"Completion-only evaluation loss: {eval_loss:.4f}; perplexity: {perplexity:.3f}")
adapter_dir = Path("computer-shop-lora-adapter")
adapter_dir.mkdir(exist_ok=True)
trainer.model.save_pretrained(adapter_dir)
tokenizer.save_pretrained(adapter_dir)
del trainer, base_model
import gc
gc.collect()
if torch.cuda.is_available(): torch.cuda.empty_cache()
fresh_base = AutoModelForCausalLM.from_pretrained(MODEL_ID, quantization_config=quant_config, torch_dtype=compute_dtype, device_map={"":0})
fine_model = PeftModel.from_pretrained(fresh_base, adapter_dir)
finetuned_answers = [ask(fine_model,q) for q in heldout_questions]
comparison_df = pd.DataFrame({"held_out_question":heldout_questions,"reference_answer":[a for q,a in eval_pairs],"original_model_answer":original_answers,"fine_tuned_model_answer":finetuned_answers})
display(comparison_df)
comparison_df.to_csv("heldout_comparison.csv",index=False)
unseen_results = [{"question":q,"original_model_answer":a,"fine_tuned_model_answer":ask(fine_model,q)} for q,a in zip(new_questions,original_new_answers)]
display(pd.DataFrame(unseen_results))
with open("unseen_prompt_comparison.json","w",encoding="utf-8") as f: json.dump(unseen_results,f,indent=2)
run_record={"model_id":MODEL_ID,"gpu":gpu_name,"gpu_memory_gib":round(gpu_memory_gib,2),"gpu_compute_capability":gpu_capability,"quantization":"4-bit NF4 QLoRA","compute_dtype":str(compute_dtype),"training_examples":len(train_pairs),"heldout_examples":len(eval_pairs),"additional_unseen_questions":len(new_questions),"evaluation_metrics":metrics,"perplexity":perplexity,"evaluation_loss_scope":"completion tokens only"}
with open("fine_tuning_run_record.json","w",encoding="utf-8") as f: json.dump(run_record,f,indent=2,default=str)
for name in ["fine_tuning_run_record.json","heldout_comparison.csv","unseen_prompt_comparison.json"]: shutil.copy(name,adapter_dir/name)
archive_path=shutil.make_archive(str(adapter_dir),"zip",root_dir=adapter_dir)
print(json.dumps(run_record,indent=2,default=str))
print("Complete adapter/results bundle:",archive_path)
from google.colab import files
files.download(archive_path)

## 6. Ask the fine-tuned model in a chat window

Run this cell after the evaluation/export cell. It launches a temporary Gradio chat link so other people can try the fine-tuned computer-shop assistant. The link is public to anyone who has it and remains available only while this Colab runtime is active.

In [ ]:
import gradio as gr
import torch

def computer_shop_chat(message, history):
    messages = [{"role": "system", "content": SYSTEM_PROMPT}]
    # Keep recent turns so the chat stays within the model context.
    for turn in (history or [])[-8:]:
        if isinstance(turn, dict) and turn.get("role") in ("user", "assistant"):
            content = turn.get("content", "")
            # Gradio 6 represents text as content blocks; older versions use strings.
            if isinstance(content, list):
                content = " ".join(
                    block.get("text", "") for block in content
                    if isinstance(block, dict) and block.get("type") == "text"
                )
            if isinstance(content, str) and content.strip():
                messages.append({"role": turn["role"], "content": content})
    messages.append({"role": "user", "content": message})
    inputs = tokenizer.apply_chat_template(
        messages, add_generation_prompt=True, return_tensors="pt", return_dict=True
    )
    inputs = {key: value.to(fine_model.device) for key, value in inputs.items()}
    prompt_length = inputs["input_ids"].shape[-1]
    fine_model.eval()
    with torch.inference_mode():
        output = fine_model.generate(
            **inputs, max_new_tokens=160, do_sample=False,
            pad_token_id=tokenizer.pad_token_id, eos_token_id=tokenizer.eos_token_id
        )
    answer = tokenizer.decode(output[0][prompt_length:], skip_special_tokens=True).strip()
    return answer or "I could not generate a response. Please try rephrasing your question."

chatbot = gr.ChatInterface(
    fn=computer_shop_chat,
    title="Computer Shop Support Chatbot",
    description="Ask the fine-tuned model about computer products and basic support. Verify important repair advice with a technician.",
    examples=["What should I consider before buying a laptop?", "Can you guarantee recovery of every deleted file?"],
    type="messages",
)
chatbot.launch(share=True, debug=False)